# Monocular Visual Odometry Coursework Notebook

This notebook contains the final cleaned implementation for the coursework.

It is organised into four parts:

1. **Setup and data loading**
2. **VO pipeline**
3. **Experiments on the provided dataset and PuzzleBot video**
4. **Noise robustness evaluation**

Recommended execution order: **Run All Cells** from top to bottom.

In [ ]:
import os
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["image.cmap"] = "gray"

## 1. Paths and calibration

The notebook expects the same directory structure used during development:

- `video/dataset_frames/` for the provided image sequence
- `video/turn_plus.mp4` for the PuzzleBot recording
- `calibration_result.npz` for the PuzzleBot camera calibration
- `vo_results/` for saved output figures

In [ ]:
PROJECT_ROOT = Path.cwd()

VIDEO_DIR = PROJECT_ROOT / "video"
RESULT_DIR = PROJECT_ROOT / "vo_results"
RESULT_DIR.mkdir(exist_ok=True)

CALIB_FILE = PROJECT_ROOT / "calibration_result.npz"
DATASET_FRAMES_DIR = VIDEO_DIR / "dataset_frames"
PUZZLEBOT_VIDEO = VIDEO_DIR / "turn_plus.mp4"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CALIB_FILE exists:", CALIB_FILE.exists())
print("DATASET_FRAMES_DIR exists:", DATASET_FRAMES_DIR.exists())
print("PUZZLEBOT_VIDEO exists:", PUZZLEBOT_VIDEO.exists())

In [ ]:
def load_calibration(npz_path: Path):
    """Load intrinsic matrix and distortion coefficients from an .npz file."""
    data = np.load(npz_path)
    print("Calibration keys:", data.files)

    if "camera_matrix" in data.files:
        K = data["camera_matrix"]
    elif "K" in data.files:
        K = data["K"]
    elif "mtx" in data.files:
        K = data["mtx"]
    else:
        raise KeyError("No camera matrix key found in calibration file.")

    if "dist_coeffs" in data.files:
        dist = data["dist_coeffs"]
    elif "dist" in data.files:
        dist = data["dist"]
    elif "distortion" in data.files:
        dist = data["distortion"]
    elif "distCoeffs" in data.files:
        dist = data["distCoeffs"]
    else:
        dist = None
        print("Warning: no distortion coefficients found.")

    return K, dist


K_puzzlebot, dist_puzzlebot = load_calibration(CALIB_FILE)
print("PuzzleBot intrinsic matrix:\n", K_puzzlebot)
print("PuzzleBot distortion coefficients:\n", dist_puzzlebot)

## 2. Data loading and preprocessing

For the provided dataset, frames are loaded from an image folder.  
For the PuzzleBot experiment, frames are extracted from the recorded video.

In [ ]:
def numeric_sort_key(path_obj: Path):
    import re
    nums = re.findall(r"\d+", path_obj.stem)
    return int(nums[-1]) if nums else path_obj.stem


def read_image_sequence(folder_path, exts=(".png", ".jpg", ".jpeg"), resize=None):
    folder_path = Path(folder_path)
    image_files = [p for p in folder_path.iterdir() if p.suffix.lower() in exts]
    image_files = sorted(image_files, key=numeric_sort_key)

    frames = []
    for p in image_files:
        img = cv2.imread(str(p))
        if img is None:
            continue
        if resize is not None:
            img = cv2.resize(img, resize)
        frames.append(img)

    return frames, image_files


def read_video_frames(video_path, max_frames=None, resize=None):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise IOError(f"Cannot open video: {video_path}")

    frames = []
    count = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if resize is not None:
            frame = cv2.resize(frame, resize)

        frames.append(frame)
        count += 1

        if max_frames is not None and count >= max_frames:
            break

    cap.release()
    return frames


def undistort_frame(frame, K, dist_coeffs):
    if K is None or dist_coeffs is None:
        return frame
    return cv2.undistort(frame, K, dist_coeffs)


def preprocess_frame(frame, K=None, dist_coeffs=None, to_gray=True, use_clahe=False):
    frame = undistort_frame(frame, K, dist_coeffs)

    if to_gray and len(frame.shape) == 3:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if use_clahe:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        frame = clahe.apply(frame)

    return frame


def show_image(img, title="Image", figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if len(img.shape) == 2:
        plt.imshow(img, cmap="gray")
    else:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
dataset_frames, dataset_files = read_image_sequence(DATASET_FRAMES_DIR)
puzzlebot_frames = read_video_frames(PUZZLEBOT_VIDEO, max_frames=300)

print("Dataset frames loaded:", len(dataset_frames))
print("PuzzleBot frames loaded:", len(puzzlebot_frames))

if len(dataset_frames) > 0:
    show_image(dataset_frames[0], "Sample Dataset Frame")

if len(puzzlebot_frames) > 0:
    show_image(puzzlebot_frames[0], "Sample PuzzleBot Frame")

## 3. Feature matching and pose estimation

Two feature pipelines are used:

- **ORB** for the provided dataset
- **SIFT** for the PuzzleBot sequence

The relative pose is estimated from matched points using the Essential Matrix and `recoverPose`.

In [ ]:
def detect_and_match_orb(img1, img2, nfeatures=2000):
    orb = cv2.ORB_create(
        nfeatures=nfeatures,
        scaleFactor=1.2,
        nlevels=8,
        edgeThreshold=15,
        patchSize=31,
        fastThreshold=10
    )

    kp1, des1 = orb.detectAndCompute(img1, None)
    kp2, des2 = orb.detectAndCompute(img2, None)

    if des1 is None or des2 is None or len(kp1) < 8 or len(kp2) < 8:
        return kp1, kp2, []

    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    matches = bf.match(des1, des2)
    matches = sorted(matches, key=lambda x: x.distance)
    return kp1, kp2, matches


def detect_and_match_sift(img1, img2, nfeatures=2000):
    sift = cv2.SIFT_create(nfeatures=nfeatures)

    kp1, des1 = sift.detectAndCompute(img1, None)
    kp2, des2 = sift.detectAndCompute(img2, None)

    if des1 is None or des2 is None or len(kp1) < 8 or len(kp2) < 8:
        return kp1, kp2, []

    index_params = dict(algorithm=1, trees=5)
    search_params = dict(checks=50)

    flann = cv2.FlannBasedMatcher(index_params, search_params)
    knn_matches = flann.knnMatch(des1, des2, k=2)

    good_matches = []
    for pair in knn_matches:
        if len(pair) < 2:
            continue
        m, n = pair
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)

    good_matches = sorted(good_matches, key=lambda x: x.distance)
    return kp1, kp2, good_matches


def draw_matches(img1, img2, kp1, kp2, matches, max_draw=80):
    matches_to_draw = matches[:max_draw]
    return cv2.drawMatches(
        img1, kp1, img2, kp2, matches_to_draw, None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )


def get_matched_points(kp1, kp2, matches):
    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])
    return pts1, pts2


def estimate_relative_pose(kp1, kp2, matches, K):
    if len(matches) < 8:
        return None, None, None, None, None

    pts1, pts2 = get_matched_points(kp1, kp2, matches)

    E, mask = cv2.findEssentialMat(
        pts1, pts2, K,
        method=cv2.RANSAC,
        prob=0.999,
        threshold=1.0
    )

    if E is None:
        return None, None, None, pts1, pts2

    _, R, t, pose_mask = cv2.recoverPose(E, pts1, pts2, K)
    return R, t, pose_mask, pts1, pts2

In [ ]:
# Quick visual check of feature matching on one frame pair

dataset_gray_1 = preprocess_frame(dataset_frames[0], None, None, to_gray=True)
dataset_gray_2 = preprocess_frame(dataset_frames[1], None, None, to_gray=True)

kp1, kp2, matches = detect_and_match_orb(dataset_gray_1, dataset_gray_2)
print("Dataset ORB matches:", len(matches))
show_image(draw_matches(dataset_gray_1, dataset_gray_2, kp1, kp2, matches, max_draw=60),
           "Dataset ORB Matches", figsize=(12, 5))

puzzle_gray_1 = preprocess_frame(puzzlebot_frames[0], K_puzzlebot, dist_puzzlebot, to_gray=True, use_clahe=True)
puzzle_gray_2 = preprocess_frame(puzzlebot_frames[1], K_puzzlebot, dist_puzzlebot, to_gray=True, use_clahe=True)

kp1_s, kp2_s, matches_s = detect_and_match_sift(puzzle_gray_1, puzzle_gray_2)
print("PuzzleBot SIFT matches:", len(matches_s))
show_image(draw_matches(puzzle_gray_1, puzzle_gray_2, kp1_s, kp2_s, matches_s, max_draw=60),
           "PuzzleBot SIFT Matches", figsize=(12, 5))

## 4. Visual odometry pipeline

The provided dataset and the PuzzleBot video are handled slightly differently:

- **Dataset**: use a default intrinsic matrix derived from image size, with no PuzzleBot undistortion
- **PuzzleBot**: use the real camera calibration and CLAHE preprocessing

In [ ]:
def make_default_intrinsics_from_frame(frame):
    h, w = frame.shape[:2]
    fx = 0.9 * w
    fy = 0.9 * w
    cx = w / 2
    cy = h / 2
    return np.array([
        [fx, 0, cx],
        [0, fy, cy],
        [0,  0,  1]
    ], dtype=np.float64)


def run_monocular_vo(frames, K_pose, K_preprocess=None, dist_coeffs=None,
                     method="orb", nfeatures=2000, step=1, use_clahe=False,
                     min_inliers_success=50):
    trajectory = []
    stats = []

    R_global = np.eye(3, dtype=np.float64)
    t_global = np.zeros((3, 1), dtype=np.float64)
    trajectory.append(t_global.flatten())

    prev_gray = preprocess_frame(
        frames[0], K=K_preprocess, dist_coeffs=dist_coeffs,
        to_gray=True, use_clahe=use_clahe
    )

    for i in range(step, len(frames), step):
        curr_gray = preprocess_frame(
            frames[i], K=K_preprocess, dist_coeffs=dist_coeffs,
            to_gray=True, use_clahe=use_clahe
        )

        if method == "orb":
            kp1, kp2, matches = detect_and_match_orb(prev_gray, curr_gray, nfeatures=nfeatures)
        elif method == "sift":
            kp1, kp2, matches = detect_and_match_sift(prev_gray, curr_gray, nfeatures=nfeatures)
        else:
            raise ValueError("method must be 'orb' or 'sift'")

        if len(matches) < 8:
            stats.append({
                "frame_idx": i,
                "num_matches": len(matches),
                "num_inliers": 0,
                "inlier_ratio": 0.0,
                "success": 0
            })
            trajectory.append(t_global.flatten())
            prev_gray = curr_gray
            continue

        R_rel, t_rel, pose_mask, pts1, pts2 = estimate_relative_pose(kp1, kp2, matches, K_pose)

        if R_rel is None or t_rel is None or pose_mask is None:
            stats.append({
                "frame_idx": i,
                "num_matches": len(matches),
                "num_inliers": 0,
                "inlier_ratio": 0.0,
                "success": 0
            })
            trajectory.append(t_global.flatten())
            prev_gray = curr_gray
            continue

        num_inliers = int(np.count_nonzero(pose_mask))
        inlier_ratio = num_inliers / max(len(matches), 1)
        success = 1 if num_inliers >= min_inliers_success else 0

        if success == 1:
            t_global = t_global + R_global @ t_rel
            R_global = R_rel @ R_global

        stats.append({
            "frame_idx": i,
            "num_matches": len(matches),
            "num_inliers": num_inliers,
            "inlier_ratio": inlier_ratio,
            "success": success
        })

        trajectory.append(t_global.flatten())
        prev_gray = curr_gray

    return np.array(trajectory), stats


def summarize_stats(stats):
    num_frames = len(stats)
    num_success = sum(s["success"] for s in stats)
    avg_matches = np.mean([s["num_matches"] for s in stats]) if stats else 0
    avg_inliers = np.mean([s["num_inliers"] for s in stats]) if stats else 0
    avg_ratio = np.mean([s["inlier_ratio"] for s in stats]) if stats else 0

    print(f"Processed frame pairs: {num_frames}")
    print(f"Successful pose estimations: {num_success}")
    print(f"Success rate: {num_success / num_frames:.3f}" if num_frames > 0 else "N/A")
    print(f"Average matches: {avg_matches:.2f}")
    print(f"Average inliers: {avg_inliers:.2f}")
    print(f"Average inlier ratio: {avg_ratio:.3f}")


def plot_trajectory_2d(trajectory, title="Estimated Trajectory"):
    plt.figure(figsize=(6, 6))
    plt.plot(trajectory[:, 0], trajectory[:, 2])
    plt.xlabel("X")
    plt.ylabel("Z")
    plt.title(title)
    plt.axis("equal")
    plt.grid(True)
    plt.show()


def save_trajectory_plot(trajectory, save_path, title="Estimated Trajectory"):
    plt.figure(figsize=(6, 6))
    plt.plot(trajectory[:, 0], trajectory[:, 2])
    plt.xlabel("X")
    plt.ylabel("Z")
    plt.title(title)
    plt.axis("equal")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.close()


def normalize_trajectory(traj):
    traj = traj.copy()
    traj = traj - traj[0]
    scale = np.linalg.norm(traj[-1])
    if scale > 0:
        traj = traj / scale
    return traj

## 5. Experiment A: provided dataset

This experiment uses ORB on the provided image sequence.  
A default intrinsic matrix is generated from frame size because the PuzzleBot calibration should not be applied to the provided dataset.

In [ ]:
K_dataset = make_default_intrinsics_from_frame(dataset_frames[0])

trajectory_ds, stats_ds = run_monocular_vo(
    dataset_frames[:500],
    K_pose=K_dataset,
    K_preprocess=None,
    dist_coeffs=None,
    method="orb",
    nfeatures=2000,
    step=1,
    use_clahe=False,
    min_inliers_success=200
)

summarize_stats(stats_ds)
plot_trajectory_2d(trajectory_ds, title="Dataset VO Trajectory (ORB)")
save_trajectory_plot(trajectory_ds, RESULT_DIR / "dataset_trajectory.png",
                     title="Dataset VO Trajectory (ORB)")

## 6. Experiment B: PuzzleBot video

This experiment evaluates the VO pipeline on the self-recorded PuzzleBot sequence.

Compared with the provided dataset, this video is much more challenging because it contains weaker texture, local motion blur, and less stable imaging conditions. In practice, this leads to fewer reliable correspondences and lower pose estimation success.

SIFT and CLAHE are used here to improve robustness, but the result is still noticeably less stable than the dataset experiment. This reflects an important limitation of classical monocular VO in real-world environments.

In [ ]:
print("PUZZLEBOT_VIDEO =", PUZZLEBOT_VIDEO)
print("Video exists:", PUZZLEBOT_VIDEO.exists())
print("Number of loaded video frames:", len(puzzlebot_frames))
print("First frame shape:", puzzlebot_frames[0].shape if len(puzzlebot_frames) > 0 else None)

In [ ]:
# 6. Experiment B: PuzzleBot video

print("Running PuzzleBot experiment...")

puzzlebot_frames = read_video_frames(PUZZLEBOT_VIDEO, max_frames=300)

trajectory_pb, stats_pb = run_monocular_vo(
    puzzlebot_frames,
    K_pose=K_puzzlebot,
    K_preprocess=K_puzzlebot,
    dist_coeffs=dist_puzzlebot,
    method="sift",
    nfeatures=4000,
    step=2,
    use_clahe=True,
    min_inliers_success=8   
)

summarize_stats(stats_pb)
plot_trajectory_2d(trajectory_pb, title="PuzzleBot VO Trajectory (SIFT + CLAHE)")
save_trajectory_plot(
    trajectory_pb,
    RESULT_DIR / "puzzlebot_trajectory.png",
    title="PuzzleBot VO Trajectory (SIFT + CLAHE)"
)

## 7. Trajectory comparison

The two trajectories are normalized before comparison because monocular VO cannot recover absolute scale.

In [ ]:
traj_ds_norm = normalize_trajectory(trajectory_ds)
traj_pb_norm = normalize_trajectory(trajectory_pb)

plt.figure(figsize=(6, 6))
plt.plot(traj_ds_norm[:, 0], traj_ds_norm[:, 2], label="Dataset")
plt.plot(traj_pb_norm[:, 0], traj_pb_norm[:, 2], label="PuzzleBot")
plt.legend()
plt.xlabel("X")
plt.ylabel("Z")
plt.title("Normalized VO Trajectory Comparison")
plt.grid(True)
plt.tight_layout()
plt.savefig(RESULT_DIR / "trajectory_comparison_normalized.png", dpi=200)
plt.show()

## 8. Noise robustness experiment

Gaussian noise is added to the provided dataset to evaluate how the VO pipeline degrades as image quality decreases.

In [ ]:
def add_gaussian_noise(image, sigma=10):
    image = image.astype(np.float32)
    noise = np.random.normal(0, sigma, image.shape)
    noisy = image + noise
    return np.clip(noisy, 0, 255).astype(np.uint8)


def make_noisy_sequence(frames, sigma):
    return [add_gaussian_noise(frame, sigma=sigma) for frame in frames]


def run_noise_experiment(frames, K_pose, K_preprocess, dist_coeffs,
                         noise_levels, nfeatures=2000, step=1,
                         method="orb", use_clahe=False,
                         min_inliers_success=50):
    results = []

    for sigma in noise_levels:
        if sigma == 0:
            test_frames = frames
        else:
            test_frames = make_noisy_sequence(frames, sigma=sigma)

        trajectory, stats = run_monocular_vo(
            test_frames,
            K_pose=K_pose,
            K_preprocess=K_preprocess,
            dist_coeffs=dist_coeffs,
            method=method,
            nfeatures=nfeatures,
            step=step,
            use_clahe=use_clahe,
            min_inliers_success=min_inliers_success
        )

        num_frames = len(stats)
        num_success = sum(s["success"] for s in stats)
        avg_matches = np.mean([s["num_matches"] for s in stats]) if stats else 0
        avg_inliers = np.mean([s["num_inliers"] for s in stats]) if stats else 0
        avg_inlier_ratio = np.mean([s["inlier_ratio"] for s in stats]) if stats else 0

        results.append({
            "sigma": sigma,
            "success_rate": num_success / num_frames if num_frames > 0 else 0,
            "avg_matches": avg_matches,
            "avg_inliers": avg_inliers,
            "avg_inlier_ratio": avg_inlier_ratio,
            "trajectory": trajectory
        })

    return results


def plot_noise_results(noise_results):
    sigmas = [r["sigma"] for r in noise_results]
    avg_matches = [r["avg_matches"] for r in noise_results]
    avg_inliers = [r["avg_inliers"] for r in noise_results]

    plt.figure(figsize=(6, 4))
    plt.plot(sigmas, avg_matches, marker="o")
    plt.xlabel("Gaussian Noise Sigma")
    plt.ylabel("Average Matches")
    plt.title("VO Robustness: Average Matches vs Noise Level")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / "noise_avg_matches.png", dpi=200)
    plt.show()

    plt.figure(figsize=(6, 4))
    plt.plot(sigmas, avg_inliers, marker="o")
    plt.xlabel("Gaussian Noise Sigma")
    plt.ylabel("Average Inliers")
    plt.title("VO Robustness: Average Inliers vs Noise Level")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULT_DIR / "noise_avg_inliers.png", dpi=200)
    plt.show()

In [ ]:
noise_results = run_noise_experiment(
    dataset_frames[:200],
    K_pose=K_dataset,
    K_preprocess=None,
    dist_coeffs=None,
    noise_levels=[0, 20, 40, 60, 80, 100],
    nfeatures=2000,
    step=1,
    method="orb",
    use_clahe=False,
    min_inliers_success=200
)

for r in noise_results:
    print(
        f"sigma={r['sigma']}, "
        f"success_rate={r['success_rate']:.3f}, "
        f"avg_matches={r['avg_matches']:.2f}, "
        f"avg_inliers={r['avg_inliers']:.2f}, "
        f"avg_inlier_ratio={r['avg_inlier_ratio']:.3f}"
    )

plot_noise_results(noise_results)

## 9. Summary

This final notebook keeps only the cleaned coursework pipeline:

- ORB on the provided dataset
- SIFT + CLAHE on the PuzzleBot video
- Gaussian noise robustness analysis
- Saved figures for the report

The main outputs are written to the `vo_results/` folder.